In [ ]:
import numpy as np
import pandas as pd

In [ ]:
movies = pd.read_csv("tmdb_5000_movies.csv/tmdb_5000_movies.csv")
credits = pd.read_csv("tmdb_5000_credits.csv/tmdb_5000_credits.csv")
# this way we created two data frames

In [ ]:
movies.head(1) # first row of the dataframe

In [ ]:
credits.head(1)

In [ ]:
credits.head(1).cast.values

In [ ]:
credits.head(1).crew.values

In [ ]:
# merging dataframes
# movies.merge(credits, on="title").shape # 23 rows, 4 from credits df and 20 from movies df and 
# one common column so total 23 columns

# new df after merging is named as movies as:
movies = movies.merge(credits, on="title")

In [ ]:
movies.head(1) # got merged dataframe

In [ ]:
# 📌content based recommendation system -> 23 columns -> deciding to drop some of them
# decide like what is used to create tags:
# genres
# id
# keywords
# title
# overview (summary)
# cast
# crew

# 🔴 no including these rn, but i want to but time constraints...as more
# runtime
# release_date
# spoken_languages

In [ ]:
movies.info() #Print a concise summary of a DataFrame. This method prints information 
# about a DataFrame including the index dtype and columns, non-null values and memory usage.

In [ ]:
# new dataframe
movies = movies[["movie_id", "title", "overview", "genres", "keywords", "cast", "crew"]]

In [ ]:
# without any paramter, this head() gave only 5 rows
movies.head() 

In [ ]:
# Check if ANY null exists in whole DataFrame 
movies.isnull().values.any() 

In [ ]:
# Count null values column-wise 
movies.isnull().sum()

In [ ]:
# check if any duplicates exists
movies.duplicated().any()

In [ ]:
# drop those rows with null values and in-place
movies.dropna(inplace=True)

In [ ]:
movies.isnull().sum()

In [ ]:
# TODO is to make a new df with id, title, tag -> tag by merging columns -> overview, genres, 
# keywords, cast and crew -> first have to clean up these 4 columns (overview too clean[mtlb
# words only for filtering (me)] hi hai)
# first genres -> list of dictionaries, say at first indexed movie, i want value of key name from
# all dictionaries of first movie(0th indexed) in a list
movies.iloc[0].genres

In [ ]:
def convert(obj):
    l = []
    for i in obj:
        l.append(i["name"]) # i is a STRING, not a dictionary
    return l

In [ ]:
# ERROR
# convert('[{"id": 28, "name": "Action"}, {"id": 12, "name": "Adventure"}, {"id": 14, "name": "Fantasy"}, {"id": 878, "name": "Science Fiction"}]')
# ⚠️ That input is NOT a list of dictionaries
# It is a STRING that looks like a list.

In [ ]:
# 🟢Correct approach: convert STRING → Python object
import json

A FLOW:
DataFrame column
   ↓
String (JSON-like)
   ↓
json.loads()
   ↓
Python list / dict
   ↓
Now access keys

In [ ]:
# json.loads() -> This function takes a JSON string and converts it into a corresponding Python 
# object, typically a dictionary.
def convert(obj):
    obj = json.loads(obj)   # string → list of dicts
    l = []
    for i in obj:
        l.append(i["name"])
    return l

In [ ]:
convert('[{"id": 28, "name": "Action"}, {"id": 12, "name": "Adventure"}, {"id": 14, "name": "Fantasy"}, {"id": 878, "name": "Science Fiction"}]')

In [ ]:
# time for all rows with this convert function with in-place 
movies["genres"] = movies["genres"].apply(convert) 
# apply(func) Apply a function along an axis of the DataFrame. , axis=0(default) means apply function to each column.

In [ ]:
movies.head() # see the genres :)

In [ ]:
# now the same for all left 3 columns (overview wala column thik hai he)
movies["keywords"] = movies["keywords"].apply(convert)

In [ ]:
# next for cast, we are extracting first 3 actors' name (real name)
def convert3(obj):
    obj = json.loads(obj)   # string → list of dicts
    l = []
    cnt = 0
    for i in obj:
        if cnt != 3:
            l.append(i["name"])
            cnt+=1
    return l

In [ ]:
movies["cast"] = movies["cast"].apply(convert3)

In [ ]:
movies.head()

In [ ]:
movies.crew.iloc[0]

In [ ]:
# filter where the job equals "Director" and then extract the name.
def fetch_directors(obj):
    obj = json.loads(obj)   # string → list of dicts
    l = []
    for i in obj:
        if i["job"] == "Director": # its Director not director
            l.append(i["name"])
            break
    return l

In [ ]:
movies["crew"] = movies["crew"].apply(fetch_directors)

In [ ]:
movies.head() # data in a structured format

In [ ]:
# string to list for overview column
# The Python split() method is a built-in string function that breaks a string into a 
# list of substrings based on a specified delimiter

movies["overview"] = movies["overview"].apply(lambda sentence: sentence.split()) # by-default whitespace

In [ ]:
movies.head()

In [ ]:
# TODO transformation to remove any whitespace in names of last 4 columns and made them a 
# single entity eg "Science Fiction" as "ScienceFiction"
# 🟢📌lambda and list comprehension used :)
movies["genres"] = movies["genres"].apply(lambda x: [i.replace(" ", "") for i in x])

In [ ]:
movies["keywords"] = movies["keywords"].apply(lambda x: [i.replace(" ", "") for i in x])
movies["cast"] = movies["cast"].apply(lambda x: [i.replace(" ", "") for i in x])
movies["crew"] = movies["crew"].apply(lambda x: [i.replace(" ", "") for i in x])

In [ ]:
movies.head()

In [ ]:
# TODO concatenate all these lists of 5 columns then convert them to string -> paragraph -> 
# tag column completed
movies["tag"] = movies["overview"] + movies["genres"] + movies["keywords"] + movies["cast"] + movies["crew"]

In [ ]:
movies.head()

In [ ]:
new_df = movies[["movie_id", "title", "tag"]]

In [ ]:
new_df.head()

In [ ]:
new_df["tag"] = new_df["tag"].apply(lambda x: " ".join(x)) # join() function is a built-in string method used 
# to combine elements of an iterable (like a list, tuple, or set) into a single string, 
# using a specified separator
# SYNTAX: separator.join(iterable)

In [ ]:
# SUGGESTION tag column to lower case
new_df["tag"] = new_df["tag"].apply(lambda x: x.lower())

In [ ]:
new_df.head()

🟢📌TEXT VECTORIZATION and Stemming using NLTK

In [ ]:
import nltk
from nltk.stem.porter import PorterStemmer
ps = PorterStemmer()

In [ ]:
# ps.stem() method we will use:
# ps.stem("loving") # love
# ps.stem("dancing") # danc

# like this, for our df we'll make a fun
def stem(text):
    y=[]
    for i in text.split(): # we saw how to do stemming on sentences
        y.append(ps.stem(i))
    return " ".join(y)


In [ ]:
new_df["tag"] = new_df["tag"].apply(stem)

In [ ]:
# Import CountVectorizer class from sklearn
# This class is used to convert text data into numerical form
from sklearn.feature_extraction.text import CountVectorizer

cv = CountVectorizer(max_features=5000, stop_words="english")
# Created an object of CountVectorizer
# Think of this as creating an empty word-count machine

# -> now every movie become a vector in 5000 dimensional space
# -> means 5000 unique words from the corpus[actual data fed to ml algos for processing(me)]
# -> and why 5000 -> ye depend krta hai, km se km words m best 
# performance niklna hai -> like if we take 10000, then
# utna hi dimensionality bd jatta hai data ka, can be problematic

In [ ]:
# 1. fit()      -> learns all unique words from the text (vocabulary)
# 2. transform() -> converts text into numbers based on word counts
vector = cv.fit_transform(new_df["tag"])

# Each row = one document (string)

In [ ]:
# Print the vocabulary (unique words)
# These words become the column names in the numeric matrix
print(cv.get_feature_names_out())

In [ ]:
# Convert sparse matrix into a normal 2D array and print it
# Rows = documents
# Columns = words (from the vocabulary above)
# Values = how many times the word appears in that document
print(vector.toarray())

📌🟢Stemming using NLTK ->
doing after learning text vectorization but should be done before CountVectorization & in text cleaning pipeline.

In [ ]:
# now next -> finding the distances between two vectors
# -> will not use euclidean distances as it fails in higher dimensions
# -> instead we use cosine distance -> angle b/w two vectors
# -> distance inversly proportional to similarity

# cosine distance of every movie with every movie

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
similarity = cosine_similarity(vector)

In [ ]:
similarity.shape # 4806*4806 -> total 4806 movies hai, like first
# movie ka 4805 movies k saath similarity (similarity score: 0 to 1 k beech hi hoti hai)
# is type se 4806 * 4806 ka ek matrix form hui hai jism diagonal saree 
# 1 hai, b/c first movie ki similarity first movie k saath 1 hi hogi
# like-wise all 4806 movies
# aur vo 5000 too unique words the jo hmn vectorise kie the(me)

In [ ]:
# MAIN FUNCTION -> recommend 5 similar movies for a specific movie

similarity[0] #-> get similarity score of first movie with all 4806 movies

In [ ]:
#  all in main function ->
# TODO Get index of the movie
# TODO Attach index with similarity score (IMPORTANT)
# TODO Sort by similarity score (highest first)
# TODO Remove itself & take TOP 5
# TODO Get actual item names (from DataFrame)
# TODO also user and movie fetching m .lower() used for a bit user exp -> 
# We use .str.lower() because:
# new_df["title"] is NOT a single string, it is a column (Series) of many strings
# And:
# Python’s .lower() works on one string
# Pandas’ .str.lower() works on every string in the column

In [ ]:
def recommend(movie):
    movie_index = new_df[new_df["title"].str.lower() == movie].index[0] # .[0] think like .index is
    # a box and we take first element -> [0][0] -> 0
    distances = similarity[movie_index]
    movies_list = sorted(
        list(enumerate(distances)),  # (index, similarity_score)
        reverse=True, # descending order
        key=lambda x: x[1]) # sorting on similarity wale pr not on index pr
    top_5 = movies_list[1:6]  
    for i in top_5:
        print(new_df.iloc[i[0]].title) 
        # i[0] -> index
        # i[1] -> similarity score

In [ ]:
# gets highest cosine similarity values (excluding itself)
user_input = input("Type: ").lower()
recommend(user_input)

DONE MODEL BUILDING ✅ -> next website dev and deployment :) 

In [ ]:
import pickle
from scipy.sparse import save_npz, csr_matrix
import numpy as np

# Convert to sparse matrix (reduces 177 MB to ~50 MB)
similarity_sparse = csr_matrix(similarity)

# Save both formats
pickle.dump(new_df, open("movies_df.pkl", "wb"))
save_npz("similarity_mat.npz", similarity_sparse)

print(f"Original size: {similarity.nbytes / 1e6:.2f} MB")
print(f"Sparse size: ~50 MB")

In [ ]:
# import pickle

In [ ]:
# pickle.dump(new_df, open("movies_df.pkl", "wb"))
# pickle.dump(similarity, open("similarity_mat.pkl", "wb"))